# 5주차 ② FashionMNIST MLP — 실습 3~9  〔빈칸본〕

> **빈칸이 4곳 있습니다.** 실습 5에 2곳, 실습 7에 2곳입니다.
> 나머지는 다 채워져 있으니 **빈칸에만 집중**하세요.
> 다 채운 노트북은 `07_mlp_fashionmnist.ipynb` 로 저장해 제출합니다.

**목표**: 입력 정규화와 가중치 초기화가 왜 필요한지 실험으로 확인하고,
`nn.Module` 로 MLP 를 정의하고 `DataLoader` 로 미니배치를 만들어
**4주차의 학습 루프 그대로** 6만 장을 학습시킨다.

> **오늘 새로 배우는 건 사실 두 개뿐입니다** — 모델을 클래스로 묶는 법, 데이터를 조각내는 법.
> **학습 루프는 4주차와 글자 그대로 같습니다.**

## 실습 3 — 정규화 유무 비교

In [ ]:
# 셀 1 — 데이터 준비
import torch, torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

ROOT = "data"          # ★ 공유 폴더를 쓰면 그 경로로 바꾼다

plain = transforms.ToTensor()                                   # 0~1 로만
norm  = transforms.Compose([transforms.ToTensor(),
                            transforms.Normalize((0.2860,), (0.3530,))])

train_plain = datasets.FashionMNIST(ROOT, train=True,  download=True, transform=plain)
train_norm  = datasets.FashionMNIST(ROOT, train=True,  download=True, transform=norm)
test_norm   = datasets.FashionMNIST(ROOT, train=False, download=True, transform=norm)

print("학습", len(train_norm), "장 / 테스트", len(test_norm), "장")
img, label = train_norm[0]
print("이미지 shape :", img.shape, "| 라벨 :", label)

> ⚠️ **다운로드가 시작되면 그대로 두세요** (약 30MB). 공유 폴더의 `data/` 를 쓰면 0초입니다.
> **`.gitignore` 에 `data/` 를 지금 추가하세요** — 2주차에 배운 것을 처음으로 실제로 쓰는 순간입니다.

> `transforms.Normalize` 는 **새 개념이 아닙니다.** 통계의 **표준화(z-score)** 와 같습니다 —
> `(값 − 평균) / 표준편차`. `(0.2860,), (0.3530,)` 은 FashionMNIST 학습셋의 실측값입니다.

In [ ]:
# 셀 2 — 정규화가 손실 곡선을 어떻게 바꾸나
def quick_train(dataset, epochs=1, lr=0.1):
    loader = DataLoader(dataset, batch_size=128, shuffle=True)
    torch.manual_seed(0)
    model = nn.Sequential(nn.Flatten(), nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 10))
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    lossfn = nn.CrossEntropyLoss()
    hist = []
    for _ in range(epochs):
        for xb, yb in loader:
            loss = lossfn(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
            hist.append(loss.item())
    return hist

h_plain = quick_train(train_plain)
h_norm  = quick_train(train_norm)

plt.plot(h_plain, label="정규화 없음", alpha=0.6)
plt.plot(h_norm,  label="Normalize 적용", alpha=0.8)
plt.xlabel("iteration"); plt.ylabel("loss"); plt.legend()
plt.title("입력 정규화의 효과 (1 epoch)"); plt.show()

print(f"마지막 50 iteration 평균 | 정규화 없음 {sum(h_plain[-50:])/50:.4f}"
      f" | 적용 {sum(h_norm[-50:])/50:.4f}")

> **관찰 포인트 ★**: 정규화한 쪽이 **더 빨리, 더 매끄럽게** 내려갑니다.
> 차이가 크지 않을 수도 있는데, **층이 깊어질수록 차이가 벌어집니다.**
> 7주차 CNN 에서 이 설정을 그대로 씁니다.

> 4주차 실습 4에서 *"`x` 가 0~10 이라 lr 1.0 이 터진다"* 고 했죠.
> **그 문제의 정식 해법이 이것**입니다. 입력 크기를 맞추면 같은 lr 이 안정적으로 동작합니다.

## 실습 4 — 가중치 초기화 3종 비교 ★

| 초기값 | 무슨 일이 |
|---|---|
| **전부 0** | 모든 뉴런이 **똑같은 값**을 갖고 똑같이 갱신된다 → 층을 넓혀도 뉴런 1개와 같다 ★ |
| **너무 큰 무작위** | 활성값이 폭주하거나(ReLU), 양끝에 몰려 기울기 소실(Sigmoid) |
| **Xavier / He** | 층의 입출력 크기에 맞춰 **적당한 크기**로 (Xavier ↔ Sigmoid·Tanh / He ↔ ReLU) |

In [ ]:
# 셀 3 — 초기값을 바꿔 본다
def make_model(kind):
    torch.manual_seed(0)
    m = nn.Sequential(nn.Flatten(), nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 10))
    for layer in m:
        if isinstance(layer, nn.Linear):
            if kind == "zero":
                nn.init.zeros_(layer.weight)
            elif kind == "big":
                nn.init.normal_(layer.weight, mean=0.0, std=1.0)     # 너무 큰 값
            elif kind == "he":
                nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")
            nn.init.zeros_(layer.bias)
    return m

loader = DataLoader(train_norm, batch_size=128, shuffle=True)
lossfn = nn.CrossEntropyLoss()

for kind in ["zero", "big", "he"]:
    model = make_model(kind)
    opt = torch.optim.SGD(model.parameters(), lr=0.1)
    losses = []
    for i, (xb, yb) in enumerate(loader):
        loss = lossfn(model(xb), yb)
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
        if i == 99: break                     # 100 iteration 만
    print(f"{kind:5s} | 첫 loss {losses[0]:7.4f} → 100번째 {losses[-1]:7.4f}")

In [ ]:
# 셀 4 — 0으로 초기화하면 뉴런이 다 똑같다
m = make_model("zero")
xb, yb = next(iter(loader))
h = m[1](m[0](xb))                            # Flatten → Linear
print("첫 층 출력의 처음 5개 뉴런 :", h[0, :5])
print("→ 전부 같은 값이면 대칭이 안 깨진 것")

# 비교: He 초기화는 뉴런마다 값이 다르다
h_he = make_model("he")[1](m[0](xb))
print("\nHe 초기화의 처음 5개 뉴런 :", h_he[0, :5])

> **결과 해석 ★★**: 0 초기화에서는 **첫 층 뉴런들이 전부 같은 값**을 냅니다.
> 같은 값을 내면 같은 기울기를 받고, 같은 기울기로 갱신되므로 **영원히 같습니다.**
> 뉴런 128개를 두었지만 **실제로는 1개짜리 층**입니다. 이게 *"대칭성이 깨지지 않는다"* 는 말입니다.

> `big`(std=1.0)은 반대 문제입니다 — 값이 커서 학습이 불안정합니다.
> **He 초기화는 층 크기에 맞춰 적당한 크기를 자동으로 정해 줍니다.**
> `nn.Linear` 의 기본값이 이미 이 계열이라, 앞으로는 **따로 안 해도 됩니다.**

## 실습 5 — `nn.Module` 로 MLP 정의 (784 → 256 → 128 → 10)

| 메서드 | 역할 | 언제 실행되나 |
|---|---|---|
| **`__init__`** | 층을 만들어 **등록**한다 (부품 목록) | `MLP()` 로 모델을 만들 때 한 번 |
| **`forward`** | 입력이 흐르는 **순서**를 쓴다 (조립 순서) | `model(x)` 를 호출할 때마다 |

In [ ]:
# 셀 5 — 모델 정의
class MLP(nn.Module):
    def __init__(self):
        super().__init__()                          # ★ 빠뜨리면 오류
        self.flatten = nn.Flatten()                 # (B,1,28,28) → (B,784)
        self.net = nn.Sequential(
            nn.Linear(784, 256), nn.ReLU(),
            # ───── 빈칸 ① : 256 → 128 층과 ReLU (한 줄) ─────

            nn.Linear(128, 10),                     # ★ 출력에 Softmax 를 붙이지 않는다
        )

    def forward(self, x):
        # ───── 빈칸 ② : flatten 을 거쳐 net 을 통과시킨 결과를 반환 ─────
        # 힌트:  return self.net(self.______(x))
        pass

model = MLP()
print(model)
print("파라미터 수 :", sum(p.numel() for p in model.parameters()))

> **관찰 포인트**: 파라미터 수가 **235,146 개**로 나옵니다
> (784×256 + 256 + 256×128 + 128 + 128×10 + 10).
> 4주차에는 2개였습니다. `optimizer` 가 왜 필요한지 여기서 체감됩니다.

> **함정 ★★**: 마지막에 **`nn.Softmax` 를 넣지 마세요.**
> 1교시에서 본 대로 `CrossEntropyLoss` 가 안에서 처리합니다. 넣으면 두 번 씌우는 것이 됩니다.

> **함정 ★**: `model.forward(x)` 라고 직접 부르지 마세요. **`model(x)`** 로 부릅니다.
> 그래야 PyTorch 내부 처리(훅, train/eval 상태 등)가 함께 동작합니다.

In [ ]:
# 셀 6 — 가짜 입력으로 shape 검산
dummy = torch.randn(4, 1, 28, 28)          # 배치 4장
out = model(dummy)
print("입력 :", dummy.shape, "→ 출력 :", out.shape)     # (4, 10) 이어야 한다

> 모델을 만들면 **학습 전에 가짜 입력으로 shape 을 먼저 확인**하세요.
> 3주차에 배운 shape 감각이 여기서 쓰입니다. 이 습관이 7주차 CNN 부터 시간을 크게 아껴 줍니다.

## 실습 6 — `Dataset`·`DataLoader` 구성

```
   6만 장을 통째로 넣으면  1 epoch 에 파라미터를 딱 1번 갱신한다
   128장씩 넣으면          1 epoch 에 469번 갱신한다   ★ 훨씬 빨리 배운다
```

| 용어 | 뜻 | 예 (6만 장, batch 128) |
|---|---|---|
| **batch** | 한 번에 넣는 묶음 | 128장 |
| **iteration** | 갱신 1회 | 469회 (= 60000 / 128, 올림) |
| **epoch** | 전체 데이터를 한 바퀴 | 469 iteration = 1 epoch |

In [ ]:
# 셀 7 — DataLoader
BATCH = 128

train_loader = DataLoader(train_norm, batch_size=BATCH, shuffle=True)     # ★ 학습은 섞는다
test_loader  = DataLoader(test_norm,  batch_size=BATCH, shuffle=False)    # ★ 평가는 안 섞는다

print("1 epoch 의 iteration 수 :", len(train_loader))

xb, yb = next(iter(train_loader))          # 배치 하나만 꺼내 본다
print("x 배치 shape :", xb.shape)          # (128, 1, 28, 28)
print("y 배치 shape :", yb.shape)          # (128,)
print("라벨 예시    :", yb[:10])

> **관찰 포인트 ★**: `y` 의 shape 이 `(128,)` 입니다 — **원-핫이 아니라 클래스 번호**입니다.
> 1교시 §4-2에서 본 그대로이고, `CrossEntropyLoss` 가 이 형태를 받습니다.

> **함정 ★**: `num_workers` 는 **건드리지 마세요(기본값 0).**
> Windows + JupyterLab 조합에서 `num_workers > 0` 은 프로세스 스폰 오류를 자주 냅니다.

In [ ]:
# 셀 8 — 이미지를 눈으로 확인
LABELS = ["티셔츠", "바지", "풀오버", "드레스", "코트",
          "샌들", "셔츠", "스니커즈", "가방", "앵클부츠"]

fig, ax = plt.subplots(1, 8, figsize=(14, 2))
for i in range(8):
    ax[i].imshow(xb[i][0], cmap="gray")
    ax[i].set_title(LABELS[yb[i]]); ax[i].axis("off")
plt.tight_layout(); plt.show()

> **관찰 포인트**: **셔츠·코트·풀오버**가 사람 눈으로도 헷갈립니다.
> 실습 9에서 모델이 정확히 그것들을 헷갈리는 것을 보게 됩니다. **지금 미리 봐 두세요.**

> `LABELS` 는 같은 폴더의 `fashion_labels.py` 에도 들어 있습니다 —
> `from fashion_labels import LABELS` 로 대신할 수 있습니다.

## 실습 7 — MLP 학습 완주 ★★

```
[4주차]                                  [5주차 = 오늘]

for epoch in range(200):                 for epoch in range(EPOCHS):
                                             for xb, yb in train_loader:      ← ★ 이 줄만 추가
    pred = model(x)                              pred = model(xb)
    loss = loss_fn(pred, y)                      loss = loss_fn(pred, yb)
    optimizer.zero_grad()                        optimizer.zero_grad()
    loss.backward()                              loss.backward()
    optimizer.step()                             optimizer.step()
```

In [ ]:
# 셀 9 — 장치·모델·손실·옵티마이저
device = "cuda" if torch.cuda.is_available() else "cpu"     # 3주차의 표준 2줄
print("사용할 장치 :", device)

model = MLP().to(device)                    # ★ 모델을 장치로
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

EPOCHS = 5          # ★ 시간이 남으면 10 으로

In [ ]:
# 셀 10 — 학습 루프
history = []

for epoch in range(EPOCHS):
    model.train()                                   # ① 학습 모드
    running, n = 0.0, 0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)       # ② 장치로

        # ───── 빈칸 ③ : 순전파와 손실 (두 줄) ─────
        # 힌트:  pred = model(__)      loss = loss_fn(__, __)


        # ───── 빈칸 ④ : 지우기 → 역전파 → 갱신 (세 줄) ─────
        # 힌트:  4주차와 글자 그대로 같다



        running += loss.item() * xb.size(0)
        n += xb.size(0)

    epoch_loss = running / n
    history.append(epoch_loss)
    print(f"epoch {epoch+1}/{EPOCHS} | 평균 loss = {epoch_loss:.4f}")

print("\n학습 완료")

> **막히면**:
> | 증상 | 원인 |
> |---|---|
> | `Expected all tensors to be on the same device` | `.to(device)` 를 빼먹었다 (3주차 실습 8) |
> | 손실이 안 줄어든다 | `zero_grad` / `step` 이 빠졌거나 순서가 틀렸다 |
> | `Expected input batch_size ... to match target` | `forward` 에서 `flatten` 을 빼먹었다 |
> | 손실이 `nan` | `lr` 이 크다. 0.1 → 0.01 로 |
> | 아주 느리다 | `device` 가 `cpu` 다. 정상이며 epoch 를 5로 두면 완주 가능 |

In [ ]:
# 셀 11 — 손실 곡선 + 가중치 저장
import os
plt.plot(range(1, len(history)+1), history, marker="o")
plt.xlabel("epoch"); plt.ylabel("평균 loss"); plt.title("학습 곡선")
plt.show()

os.makedirs("models", exist_ok=True)
torch.save(model.state_dict(), "models/mlp_fashion.pt")     # ★ 6주차에 다시 쓴다
print("저장 완료 : models/mlp_fashion.pt")

> **포인트 ★**: `state_dict()` 는 **가중치만** 저장합니다. 모델 구조는 코드가 갖고 있습니다.
> **6주차에 이 파일을 불러와** 이어서 학습하는 법(체크포인트)을 배웁니다.
> `models/` 도 **`.gitignore` 에 추가**하세요 — 가중치는 코드가 아닙니다.

## 실습 8 — 평가·예측 확인

> **`model.eval()` 과 `torch.no_grad()` 는 다른 것입니다.**
> - `eval()` : 층의 **동작 모드**를 바꾼다 (드롭아웃·배치정규화)
> - `no_grad()` : **그래프를 안 그린다** (메모리·속도)
> 평가할 때는 **둘 다** 씁니다.

In [ ]:
# 셀 12 — 테스트 정확도
model.eval()                                    # ★ 평가 모드
correct, total = 0, 0

with torch.no_grad():                           # ★ 그래프를 안 그린다
    for xb, yb in test_loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        pred = logits.argmax(dim=1)             # ★ 가장 큰 로짓의 인덱스 = 예측 클래스
        correct += (pred == yb).sum().item()
        total += yb.size(0)

print(f"테스트 정확도 : {correct/total*100:.2f}%  ({correct}/{total})")

> **관찰 포인트 ★**: **85% 내외**가 나오면 정상입니다. 10개 클래스이므로 찍으면 10% 입니다.
> 여러분이 만든 모델이 **찍기보다 8배 이상 잘합니다.**

> `argmax(dim=1)` 은 10개 로짓 중 **가장 큰 것의 번호**를 고릅니다.
> **Softmax 를 씌워도 순서는 그대로**이므로 예측에는 로짓만으로 충분합니다.

In [ ]:
# 셀 13 — 한 장을 확률로 본다
xb, yb = next(iter(test_loader))
xb, yb = xb.to(device), yb.to(device)
with torch.no_grad():
    prob = torch.softmax(model(xb[:1]), dim=1)[0]       # ★ 여기서만 Softmax

plt.imshow(xb[0][0].cpu(), cmap="gray"); plt.axis("off")
plt.title(f"정답: {LABELS[yb[0]]}"); plt.show()

for i in prob.argsort(descending=True)[:3]:
    print(f"  {LABELS[i]:8s} {prob[i].item()*100:5.1f}%")

> **관찰 포인트**: 상위 3개 확률을 보면 **모델이 무엇과 헷갈렸는지** 보입니다.
> 1등이 90% 이상이면 확신, 40%대면 헷갈린 것입니다.

## 실습 9 — 잘못 맞힌 이미지 보기

In [ ]:
# 셀 14 — 틀린 것만 모아 본다
model.eval()
wrong_x, wrong_p, wrong_y = [], [], []

with torch.no_grad():
    for xb, yb in test_loader:
        xb, yb = xb.to(device), yb.to(device)
        pred = model(xb).argmax(dim=1)
        bad = pred != yb
        if bad.any():
            wrong_x.append(xb[bad].cpu()); wrong_p.append(pred[bad].cpu()); wrong_y.append(yb[bad].cpu())
        if sum(len(w) for w in wrong_x) >= 9:
            break

wx = torch.cat(wrong_x)[:9]; wp = torch.cat(wrong_p)[:9]; wy = torch.cat(wrong_y)[:9]

fig, ax = plt.subplots(3, 3, figsize=(7, 7))
for i, a in enumerate(ax.flat):
    a.imshow(wx[i][0], cmap="gray"); a.axis("off")
    a.set_title(f"예측 {LABELS[wp[i]]}\n정답 {LABELS[wy[i]]}", fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# 셀 15 (여유가 있으면) — 어떤 클래스에서 가장 많이 틀리나
model.eval()
per_class_wrong = torch.zeros(10)
per_class_total = torch.zeros(10)

with torch.no_grad():
    for xb, yb in test_loader:
        xb, yb = xb.to(device), yb.to(device)
        pred = model(xb).argmax(dim=1)
        for c in range(10):
            m = yb == c
            per_class_total[c] += m.sum().item()
            per_class_wrong[c] += (pred[m] != c).sum().item()

print("클래스별 오답률")
order = (per_class_wrong / per_class_total).argsort(descending=True)
for c in order:
    rate = per_class_wrong[c] / per_class_total[c] * 100
    print(f"  {LABELS[c]:8s} {rate:5.1f}%  ({int(per_class_wrong[c])}/{int(per_class_total[c])})")

> **관찰 포인트 ★★**: **셔츠 ↔ 코트 ↔ 풀오버** 조합이 압도적으로 많습니다.
> 실습 6에서 **사람 눈으로도 헷갈렸던 그 클래스들**입니다.

> **핵심 메시지 ★**: 모델의 실수는 **아무렇게나 나지 않습니다.** 사람이 헷갈리는 것을 헷갈립니다.
> *"정확도 85%"* 라는 숫자 하나보다 **어디서 틀리는지**가 훨씬 많은 것을 알려 줍니다.
> **6주차에 혼동행렬로 이것을 정식으로 봅니다.**

> **6주차로 넘기는 다리**: MLP 는 이미지를 **784개 숫자를 일렬로 늘어놓은 것**으로 봅니다.
> 옆 픽셀이 붙어 있다는 정보가 사라졌죠. **그 구조를 살리는 모델이 7주차 CNN 입니다.**

---

### 과제 (마감 10/6 화 23:59 — 6주차가 **10/7 수요일**이라 수업 전날)

```
  ① 07_mlp_fashionmnist.ipynb  ← 셀 1~15 실행, 출력 저장 상태로
  ② 테스트 정확도
  ③ 오분류 이미지 9장
  ④ 은닉층 크기나 epoch 를 한 번 바꿔 본 결과 1건   ★ 이번 과제의 핵심
  ⑤ 회고 3줄
  ⑥ 커밋 · push · LMS 제출
```

```powershell
Set-Location $env:DL2026_HOME
.\venv\Scripts\Activate.ps1
git add .
git commit -m "week5: MLP on FashionMNIST"
git push
```

> ⚠️ **`.gitignore` 에 `data/` 와 `models/` 가 들어 있는지 반드시 확인**하세요.

### 이 노트북 체크리스트

- [ ] `Normalize` 가 통계의 표준화와 같다는 것을 안다
- [ ] **0으로 초기화하면 뉴런이 전부 같아지는 것**을 출력으로 확인했다 ★
- [ ] `nn.Module` 의 `__init__` / `forward` 역할을 말할 수 있다
- [ ] MLP 파라미터 수가 약 23만 개인 것을 봤다
- [ ] 배치 shape `(128,1,28,28)` 과 `(128,)` 을 확인했다
- [ ] 6만 장 · batch 128 → **469 iteration = 1 epoch** 을 계산할 수 있다 ★
- [ ] 학습 루프가 4주차와 **배치 for 문 하나 차이**임을 안다 ★
- [ ] epoch 별 손실이 줄어드는 것을 끝까지 봤다 ★★
- [ ] `model.eval()` 과 `no_grad()` 의 차이를 말할 수 있다
- [ ] 테스트 정확도 85% 내외를 얻었다
- [ ] `state_dict()` 로 가중치를 저장했다
- [ ] 오분류 9장을 보고 **셔츠·코트·풀오버 혼동**을 확인했다 ★